<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Testing Python Code with Pytest

## Why Do We Test Code?

Imagine you build a data pipeline that calculates monthly revenue for your finance team. It works perfectly today. Next week, a colleague changes how dates are parsed. Suddenly, the revenue numbers are wrong, but nobody notices for three days.

**Testing prevents this.** Tests are small scripts that automatically check if your code still does what it's supposed to do, every time someone makes a change. Think of them as automated quality checks, the same way a data validation step catches nulls or duplicates before loading into a warehouse, tests catch bugs in your code before they reach production.

**In data engineering, testing matters because:**

- A small bug in a transformation can silently corrupt millions of rows
- Pipelines run unattended, there's no human eyeballing the output each time
- Tests give you confidence to refactor, optimise, and deploy without fear

---

## What is Pytest?

**Pytest** is the most popular testing framework in Python. It's simple, powerful, and used across the industry, from startups to large enterprises.

Why pytest over alternatives?

- Minimal boilerplate, just write functions that start with `test_`
- Excellent error messages, when something fails, it tells you *exactly* what went wrong
- Fixtures, reusable setup code (we'll cover this later)
- Huge ecosystem of plugins



To install it just:

```python

pip install pytest

```

---

## Module 1: How `assert` Works

Before we write any tests, let's understand the one keyword that powers all of them: **`assert`**.

`assert` is built into Python. It checks whether an expression is `True`. If it is, nothing happens, the programme moves on. If it's `False`, Python raises an `AssertionError` and tells you what went wrong.

**Example:**

In [ ]:
# Try these in a Python console or notebook:

assert 1 + 1 == 2                       # True → nothing happens
assert "hello".upper() == "HELLO"       # True → fine
assert 10 > 5                           # True → fine

# This one would FAIL:
# assert 1 + 1 == 3                     # False → AssertionError!

You can attach a message that shows when the assertion fails. This is very helpful for debugging:

In [ ]:
revenue = -500
assert revenue > 0, f"Revenue should be positive, got {revenue}"
# AssertionError: Revenue should be positive, got -500

AssertionError: Revenue should be positive, got -500

### Common Assert Patterns

```python
# Equality
assert result == 42

# Inequality
assert status != "failed"

# Comparison
assert count > 0

# Membership, is this item in a list?
assert "customer_id" in column_list

# Type checking
assert isinstance(my_data, list)

# Truthiness, is this collection non-empty?
assert len(results) > 0
```

That's it. Every test you'll ever write with pytest is just a function full of `assert` statements. If all asserts pass, the test passes. If any single assert fails, the test fails.

---

## Module 2: Your First Test, Anatomy of a Test Function

### The Rules

Pytest discovers and runs your tests automatically, but it needs a few naming conventions:

1. **Test files** must start with `test_` (e.g. `test_calculations.py`)
2. **Test functions** must start with `test_` (e.g. `def test_addition():`)
3. Each test function should test **one thing** and have a clear name describing what it checks

### The Pattern: Arrange → Act → Assert

Every test follows the same three-step structure. Think of it like preparing a recipe:

| Step | What it does | Analogy |
|------|-------------|---------|
| **Arrange** | Set up your test data and inputs | Gather your ingredients on the counter |
| **Act** | Run the function you want to test | Cook the dish |
| **Assert** | Check the result is what you expected | Taste it, does it match the recipe? |

### A Simple Example

Let's say we have a file called `helpers.py` with a function:

In [ ]:
# helpers.py

def add(a, b):
    """Adds two numbers together."""
    return a + b

We write a **separate** test file called `test_helpers.py`:


In [ ]:
# test_helpers.py



In [ ]:
!pip install ipytest -q
import ipytest
ipytest.autoconfig()

In [ ]:
%%ipytest -v
def add(a, b):
    """Adds two numbers together."""
    return a + b



======================================= test session starts =======================================
platform win32 -- Python 3.10.2, pytest-9.0.2, pluggy-1.6.0
rootdir: c:\Users\MiguelAngelSanchezRa\Python_Sandbox\CBS_Python_eng\05_01_Testing_Fundamentals_PyTest
collected 3 items

t_caf26424a55946e4b30b6b07d447e9f3.py ...                                                    [100%]

======================================== 3 passed in 0.03s ========================================


That's three tests. Each one calls `add()` with specific inputs and checks the output. Notice how each test name tells you exactly what scenario it covers.

### What a Test Function Has

Every test function:

- **Name** starts with `test_`, this is how pytest finds it
- **No parameters** (for now, we'll see fixtures later which change this)
- **Contains at least one `assert`**, this is what actually checks the result
- **Is independent**, each test should work on its own, without depending on other tests

### More General Examples

Let's test a few more basic functions before we get into data-specific code:

In [ ]:
# string_utils.py
def clean_name(name):
    """Strips whitespace and converts to title case."""
    return name.strip().title()

def is_valid_email(email):
    """Very basic email check: must contain @ and a dot after it."""
    if "@" not in email:
        return False
    parts = email.split("@")
    return "." in parts[1]

def calculate_discount(price, discount_pct):
    """Applies a percentage discount. Returns the final price."""
    return round(price * (1 - discount_pct / 100), 2)

In [ ]:
# test_string_utils.py


ModuleNotFoundError: No module named 'string_utils'

### Exercise 2.1: Write Your Own Tests

Given these functions:


In [ ]:
# maths_utils.py

def multiply(a, b):
    return a * b

def is_even(n):
    return n % 2 == 0

Write tests for:

1. `multiply(4, 5)` should return `20`
2. `multiply(-3, 3)` should return `-9`
3. `multiply(anything, 0)` should return `0`
4. `is_even(4)` should be `True`
5. `is_even(7)` should be `False`

In [ ]:
##Solution:
# test_maths_utils.py


## Module 3: Running Pytest from the Terminal

### The Setup: Two Files Side by Side

In a real project you have your **source code** in one file and your **tests** in another. Pytest scans for files starting with `test_` and runs every function inside them that also starts with `test_`.

```
my_project/
├── helpers.py            ← Your functions live here
└── test_helpers.py       ← Your tests live here
```

Or, in a more structured project:

```
my_project/
├── src/
│   ├── helpers.py
│   └── preprocessing.py
├── tests/
│   ├── test_helpers.py
│   └── test_preprocessing.py
└── main.py
```

### How to Run Tests

Open your terminal, navigate to your project folder, and run:

```bash
# Run ALL tests (pytest auto-discovers files starting with test_)
pytest

# Verbose mode, shows each test name and pass/fail status
pytest -v

# Run a specific test file
pytest test_helpers.py
pytest tests/test_preprocessing.py

# Run a single test function within a file
pytest test_helpers.py::test_add_positive_numbers

# Run only tests whose names match a keyword
pytest -k "email"           # Runs test_valid_email, test_invalid_email_no_at, etc.
pytest -k "discount"        # Runs test_discount_20_percent, test_discount_zero

# Stop at the first failure (useful for debugging one thing at a time)
pytest -x

# Show print() statements in the output (normally hidden)
pytest -s

# Combine flags, verbose, stop on first failure, show prints
pytest -v -x -s
```

### Reading the Output

When you run `pytest -v`, the output looks like this:

```
test_helpers.py::test_add_positive_numbers PASSED
test_helpers.py::test_add_negative_numbers PASSED
test_helpers.py::test_add_zero PASSED
test_string_utils.py::test_clean_name_strips_whitespace PASSED
test_string_utils.py::test_discount_20_percent FAILED

FAILED test_string_utils.py::test_discount_20_percent
    assert calculate_discount(100, 20) == 85.0
    AssertionError: assert 80.0 == 85.0

========================= 1 failed, 4 passed =========================
```

The key information: which tests passed, which failed, and for failures, **what the actual vs expected value was**. Pytest's error messages are one of its best features, they show you exactly what went wrong.

### Quick Reference: CLI Flags

| Command | Purpose |
|---------|---------|
| `pytest` | Run all tests |
| `pytest -v` | Verbose, show each test name |
| `pytest -x` | Stop on first failure |
| `pytest -s` | Show print statements |
| `pytest -k "keyword"` | Run tests matching keyword |
| `pytest tests/test_file.py` | Run specific file |
| `pytest file.py::test_func` | Run specific test function |
| `pytest -v -x -s` | Combine flags |

---

## Module 4: Testing Data Transformations with Pandas

Now let's apply what we've learned to the kind of functions you'll actually write in data pipelines.


In [ ]:
# preprocessing.py
import pandas as pd
import numpy as np

def convert_price_column(df):
    """Converts a 'price' column from string to numeric. Non-numeric values become NaN."""
    df = df.copy()
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    return df

def fill_missing_prices(df, fill_value=0.0):
    """Fills NaN values in the 'price' column with a default."""
    df = df.copy()
    df['price'] = df['price'].fillna(fill_value)
    return df

def add_total_column(df):
    """Adds a 'total' column = quantity * price."""
    df = df.copy()
    df['total'] = df['quantity'] * df['price']
    return df

def categorise_order_size(df):
    """
    Adds an 'order_size' column:
    - 'Small' if quantity <= 3
    - 'Medium' if quantity <= 10
    - 'Large' if quantity > 10
    """
    df = df.copy()
    conditions = [df['quantity'] <= 3, df['quantity'] <= 10, df['quantity'] > 10]
    choices = ['Small', 'Medium', 'Large']
    df['order_size'] = np.select(conditions, choices, default='Unknown')
    return df


### Testing Each Function

In [ ]:
# test_preprocessing.py


### Exercise 4.1: Write Your Own Tests

Given the functions above, write tests for:

1. What happens when `add_total_column` receives a DataFrame where `quantity` is zero?
2. What happens when `convert_price_column` receives a DataFrame where ALL prices are valid numbers? (No NaN should be created.)
3. Does `categorise_order_size` assign `'Large'` to a quantity of exactly `11`?

In [ ]:
##Solutions: 


## Module 5: Fixtures, Reusable Test Data

### The Problem

Notice how we keep creating DataFrames in every test? That's repetitive. What if 10 tests all need the same sample data?

**Fixtures** solve this. A fixture is a function decorated with `@pytest.fixture` that provides test data to your tests. Pytest automatically passes it in when it sees the fixture name as a function parameter.

Think of a fixture like a template dataset that you prepare once and hand to any test that needs it, each test gets its own fresh copy, so they never interfere with each other.

### Defining and Using Fixtures

In [ ]:
# test_preprocessing.py

import pytest
import pandas as pd
import numpy as np
from preprocessing import convert_price_column, add_total_column

@pytest.fixture
def sample_orders():
    """A small potions shop dataset with known edge cases."""
    return pd.DataFrame({
        'order_id': ['ORD001', 'ORD002', 'ORD003', 'ORD004'],
        'potion_name': ['Healing', 'Invisibility', 'Strength', 'Healing'],
        'quantity': [2, 15, 5, 1],
        'price': ['12.50', '45.00', ' ', '12.50'],   # Note: space in row 3
        'region': ['North', 'South', 'North', 'East']
    })

@pytest.fixture
def clean_orders():
    """Pre-cleaned numeric data, for tests that don't need to test cleaning."""
    return pd.DataFrame({
        'order_id': ['ORD001', 'ORD002', 'ORD003'],
        'quantity': [2, 15, 5],
        'price': [12.50, 45.00, 8.00],
        'region': ['North', 'South', 'North']
    })


# --- Tests using fixtures ---
# Notice: the parameter name matches the fixture function name.
# Pytest sees 'sample_orders' in the signature and injects the return value.

def test_convert_price_with_fixture(sample_orders):
    result = convert_price_column(sample_orders)
    assert pd.api.types.is_numeric_dtype(result['price'])
    assert np.isnan(result['price'][2])  # The space became NaN

def test_add_total_with_fixture(clean_orders):
    result = add_total_column(clean_orders)
    assert result['total'][0] == 25.0    # 2 * 12.50
    assert result['total'][1] == 675.0   # 15 * 45.00

### How Fixtures Work, The Mechanism

```
1. Pytest sees:  def test_something(sample_orders)
2. It looks for a @pytest.fixture named 'sample_orders'
3. It runs that fixture function
4. It passes the return value into your test
5. Each test gets a FRESH copy, tests never affect each other
```

### The conftest.py File

In real projects, you put shared fixtures in a special file called **`conftest.py`**. Any test file in the same directory (or below) can use those fixtures automatically, **no import needed**.

```
my_project/
├── src/
│   ├── preprocessing.py
│   └── io_handler.py
├── tests/
│   ├── conftest.py              ← Shared fixtures live here
│   ├── test_preprocessing.py    ← Can use fixtures from conftest.py
│   └── test_io_handler.py       ← Can also use them, no import!
└── main.py
```


In [ ]:
# tests/conftest.py

import pytest
import pandas as pd
import numpy as np

@pytest.fixture
def sample_orders():
    """Shared test data available to ALL test files automatically."""
    return pd.DataFrame({
        'order_id': ['ORD001', 'ORD002', 'ORD003'],
        'quantity': [2, 15, 5],
        'price': ['12.50', '45.00', ' '],
        'region': ['North', 'South', 'North']
    })

@pytest.fixture
def pipeline_config():
    """Shared configuration for pipeline tests."""
    return {
        "target": "is_premium",
        "numeric_cols": ["quantity", "price"],
        "categorical_cols": ["region"],
    }

**Key Point:** You never write `from conftest import ...`. Pytest discovers `conftest.py` automatically. Just name the file `conftest.py` and place it in your tests folder.

### Exercise 5.1: Create a Fixture

Write a fixture called `messy_sales` that returns a DataFrame with these characteristics:

- 4 rows
- Columns: `product`, `quantity`, `price`
- At least one `price` value that is a string space `' '`
- At least one `quantity` that is `0`

Then write one test that uses it.

In [ ]:
##Solution: 


## Module 6: Testing for Errors, When Things *Should* Break

Good code raises informative errors when given bad input. We need to test that too.

### Simple Error Testing with `pytest.raises`

`pytest.raises` lets you assert that a function **does** raise a specific exception. If the function *doesn't* raise, the test **fails**.

Let's start with a simple example:

In [ ]:
# calculations.py

def divide(a, b):
    """Divides a by b. Raises ValueError if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

In [ ]:
# test_calculations.py

import pytest
from calculations import divide



Think of it this way: you're testing the **guardrails**, not the happy path. You *want* the error to happen, that means your protection is working.

### Testing More Complex Errors, File Not Found

A very common real-world case: what happens when a file doesn't exist?

In [ ]:
# io_handler.py

import pandas as pd
import os

def load_csv(filepath):
    """Reads a CSV file. Raises FileNotFoundError if path doesn't exist."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    return pd.read_csv(filepath)

In [ ]:
# test_io_handler.py

import pytest
from io_handler import load_csv

def test_load_csv_file_not_found():
    with pytest.raises(FileNotFoundError):
        load_csv("this/path/does/not/exist.csv")

def test_load_csv_error_message():
    with pytest.raises(FileNotFoundError, match="File not found"):
        load_csv("fake_data.csv")

### Testing File I/O with `tmp_path`

Pytest provides a built-in fixture called `tmp_path`, it gives you a temporary directory that is automatically created before the test and cleaned up after. This is essential for testing read/write operations without polluting your real file system.

In [ ]:
# io_handler.py (continued)

def save_to_csv(df, filepath):
    """Saves a DataFrame to CSV, creating directories if needed."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    df.to_csv(filepath, index=False)

In [ ]:
# test_io_handler.py (continued)

import pandas as pd
import os

def test_save_creates_file(tmp_path):
    """tmp_path gives us a safe temporary directory."""
    df = pd.DataFrame({'a': [1, 2], 'b': [3, 4]})
    output_file = tmp_path / "output" / "results.csv"

    save_to_csv(df, str(output_file))

    assert os.path.exists(output_file)

def test_save_and_reload_integrity(tmp_path):
    """Write → Read → Compare. Data should survive the round trip."""
    df = pd.DataFrame({'name': ['Alice', 'Bob'], 'score': [95, 82]})
    filepath = tmp_path / "round_trip.csv"

    save_to_csv(df, str(filepath))
    loaded = pd.read_csv(filepath)

    assert len(loaded) == 2
    assert list(loaded.columns) == ['name', 'score']
    assert loaded['name'][0] == 'Alice'

## Module 7: Approximate Comparisons with `pytest.approx`

Floating-point arithmetic can produce tiny rounding differences. `pytest.approx` handles this gracefully.

### The Problem

In [1]:
# Try this in Python:
result = 0.1 + 0.2
print(result)        # 0.30000000000000004
print(result == 0.3) # False!

0.30000000000000004
False


This is how all computers handle decimals, it's not a bug. But it means `assert result == 0.3` would fail even though the answer is essentially correct.

### Testing Functions with Decimal Results

The key idea: define a **function** that produces a decimal result, then use `pytest.approx` to test it safely.

In [ ]:
# calculations.py

def calculate_charge_ratio(total_charges, tenure):
    """Calculates average charges per tenure period."""
    return total_charges / (tenure + 1)

def apply_tax(price, tax_rate=0.20):
    """Applies tax to a price."""
    return price * (1 + tax_rate)

def compute_average(values):
    """Returns the mean of a list of numbers."""
    return sum(values) / len(values)

In [ ]:
# test_calculations.py
import pytest
from calculations import calculate_charge_ratio, apply_tax, compute_average

def test_charge_ratio():
    result = calculate_charge_ratio(29.85, 1)
    assert result == pytest.approx(14.925)



## Module 8: Putting It All Together, A Mini Pipeline Test

Let's combine everything we've learned into a small end-to-end example. This is a simplified version of how test suites look in production projects.

### The Pipeline Function

In [ ]:
# pipeline.py

import pandas as pd
import numpy as np

def run_mini_pipeline(df, target_col):
    """
    A small cleaning pipeline that:
    1. Converts price to numeric
    2. Fills missing prices with the column mean
    3. Adds a total column (quantity * price)
    4. Maps the target column to binary (Yes=1, No=0)
    """
    df = df.copy()
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df['price'] = df['price'].fillna(df['price'].mean())
    df['total'] = df['quantity'] * df['price']
    df[f'{target_col}_binary'] = df[target_col].map({'Yes': 1, 'No': 0}).astype(int)
    return df

In [ ]:
# test_pipeline.py


## Typical Project Structure

When you're working on a real pipeline project, this is how the files are typically organised:

```
my_pipeline_project/
│
├── src/                        # Your pipeline code
│   ├── __init__.py
│   ├── config.py               # Constants (paths, column names)
│   ├── io_handler.py           # Read/write functions
│   ├── preprocessing.py        # Cleaning & transformation
│   └── pipeline.py             # Orchestrator that calls everything
│
├── tests/                      # Your tests
│   ├── conftest.py             # Shared fixtures (auto-discovered)
│   ├── test_io_handler.py      # Tests for io_handler.py
│   ├── test_preprocessing.py   # Tests for preprocessing.py
│   ├── test_pipeline.py        # Integration tests
│   └── test_main.py            # Tests for the entry point
│
├── data/
│   ├── raw/
│   └── processed/
│
├── main.py                     # Entry point
└── requirements.txt
```

Each source file has a corresponding test file. Shared test data lives in `conftest.py`. You run `pytest -v` from the project root and everything is discovered automatically.

---

## Quick Reference: Pytest Cheatsheet

### Core Syntax

| Pattern | Purpose |
|---------|---------|
| `def test_something():` | Define a test (must start with `test_`) |
| `assert x == y` | Check equality |
| `assert x != y` | Check inequality |
| `assert x > 0` | Check condition |
| `assert "word" in text` | Check membership |
| `assert isinstance(x, list)` | Check type |

### Fixtures

| Pattern | Purpose |
|---------|---------|
| `@pytest.fixture` | Define reusable test data |
| `def test_x(my_fixture):` | Use a fixture (name = parameter) |
| `conftest.py` | Auto-discovered shared fixtures file |
| `tmp_path` | Built-in fixture: temporary directory |

### Error Testing

| Pattern | Purpose |
|---------|---------|
| `with pytest.raises(ValueError):` | Expect a specific exception |
| `pytest.raises(X, match="msg")` | Expect exception with specific message |

### Approximate Comparisons

| Pattern | Purpose |
|---------|---------|
| `assert x == pytest.approx(y)` | Float comparison with default tolerance |
| `pytest.approx(y, abs=0.01)` | Custom absolute tolerance |
| `pytest.approx(y, rel=0.05)` | Custom relative tolerance (5%) |

### Pandas-Specific Assertions

| Pattern | Purpose |
|---------|---------|
| `assert len(df) == 3` | Check row count |
| `assert 'col' in df.columns` | Check column exists |
| `assert list(df.columns) == [...]` | Check all columns and order |
| `assert df['col'].isna().sum() == 0` | No missing values |
| `assert pd.api.types.is_numeric_dtype(df['x'])` | Check column data type |
| `assert isinstance(df, pd.DataFrame)` | Check it's a DataFrame |
| `assert df['col'].tolist() == [1, 2, 3]` | Check exact row values |
| `assert (df['col'] >= 0).all()` | Check all values meet a condition |

### CLI Commands

| Command | Purpose |
|---------|---------|
| `pytest` | Run all tests |
| `pytest -v` | Verbose, show each test name |
| `pytest -x` | Stop on first failure |
| `pytest -s` | Show print statements |
| `pytest -k "keyword"` | Run tests matching keyword |
| `pytest tests/test_file.py` | Run a specific file |
| `pytest file.py::test_func` | Run a specific test function |
| `pytest -v -x -s` | Combine flags |

---

## Summary: What You've Learned

| Module | Concept | Key Takeaway |
|--------|---------|-------------|
| 1 | `assert` | The foundation, checks if something is True |
| 2 | Test functions | Must start with `test_`, follow Arrange → Act → Assert |
| 3 | Running from terminal | `pytest -v` to run, flags to control behaviour |
| 4 | Data transformation tests | Test each pipeline step individually with pandas |
| 5 | Fixtures & conftest.py | Reusable test data, auto-discovered by pytest |
| 6 | `pytest.raises` & `tmp_path` | Test errors and file operations safely |
| 7 | `pytest.approx` | Handle floating-point comparisons |
| 8 | Pipeline integration | Smoke tests that verify end-to-end flow |


**Next Step:** Try reading the test files from the churn pipeline project. You should now be able to understand each test, the fixtures in `conftest.py`, the unit tests in `test_preprocessing.py`, the file I/O tests in `test_io_handler.py`, and the integration test in `test_pipeline.py`. They use exactly the same patterns you've practised here.

---

## Further Reading

Here are some resources to deepen your understanding of testing in Python:

**Official Documentation:**

- [Pytest Official Documentation](https://docs.pytest.org/en/stable/), The definitive reference. Start with the "Getting Started" section.
- [Pytest Fixtures Reference](https://docs.pytest.org/en/stable/how-to/fixtures.html), Deep dive into fixtures, scopes, and parametrisation.

**Tutorials & Guides:**

- [Real Python, Testing with Pytest](https://realpython.com/pytest-python-testing/), Beginner-friendly walkthrough with practical examples.
- [Real Python, Effective Python Testing](https://realpython.com/python-testing/), Broader overview covering testing philosophy and best practices.
- [Test-Driven Development with Python (book)](https://www.obeythetestinggoat.com/), Free online book that teaches TDD from scratch with a web project.

**Pandas-Specific Testing:**

- [Pandas Testing Utilities](https://pandas.pydata.org/docs/reference/testing.html), Built-in functions like `pd.testing.assert_frame_equal()` for comparing DataFrames in tests.
- [Testing Data Pipelines (Dataquest)](https://www.dataquest.io/blog/unit-tests-python/), Practical guide focused on data workflows.

**Advanced Topics (for later):**

- [Pytest Parametrize](https://docs.pytest.org/en/stable/how-to/parametrize.html), Run the same test with multiple inputs automatically.
- [unittest.mock (Python docs)](https://docs.python.org/3/library/unittest.mock.html), Mocking and patching for isolating units of code.
- [Coverage.py](https://coverage.readthedocs.io/), Measure how much of your code is covered by tests. Run with `pytest --cov`.